In [1]:
# Cell 1 - Install required packages
!pip install langchain langchain-groq langsmith python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.0 MB/s eta 0:00:00


In [3]:
# Cell 2 - Import and configure
import os
import json
from google.colab import userdata

# Get keys from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

# LangSmith tracing configuration (MANDATORY)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume_screening_groq_feb2026"

# Verify setup
print("Groq API Key:", "Loaded" if os.environ["GROQ_API_KEY"] else "Missing")
print("LangSmith Key:", "Loaded" if os.environ["LANGCHAIN_API_KEY"] else "Missing")
print("LangSmith Tracing:", os.environ["LANGCHAIN_TRACING_V2"])
print("Project:", os.environ["LANGCHAIN_PROJECT"])

Groq API Key: Loaded
LangSmith Key: Loaded
LangSmith Tracing: true
Project: resume_screening_groq_feb2026


In [13]:
# Cell 3 - LangChain imports (FIXED - removed pydantic_v1)
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field  # Direct import from pydantic
from typing import List, Optional

print(" All imports successful!")

 All imports successful!


In [14]:
# Cell 4 - Initialize Groq model (FREE!)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # Best free model on Groq
    temperature=0,  # Consistent outputs for scoring
    max_tokens=2048,
    timeout=None,
    max_retries=2,
)

print("Groq LLM initialized with Llama 3.3 70B")
print("(FREE tier: 30 requests per minute)")

Groq LLM initialized with Llama 3.3 70B
(FREE tier: 30 requests per minute)


In [15]:
# Cell 5 - Define output structure using Pydantic (FIXED)
class ExtractedSkills(BaseModel):
    skills: List[str] = Field(description="Technical and soft skills from resume")
    experience_years: float = Field(description="Total years of experience")
    tools: List[str] = Field(description="Tools, frameworks, technologies mentioned")

class MatchResult(BaseModel):
    matched_skills: List[str] = Field(description="Skills that match job requirements")
    missing_skills: List[str] = Field(description="Required skills missing from resume")
    experience_fit: str = Field(description="'good', 'low', or 'exceeds'")

class ScoreResult(BaseModel):
    score: int = Field(description="Fit score from 0-100")
    reasoning: str = Field(description="Detailed explanation of the score")
    recommendation: str = Field(description="'Strong Hire', 'Consider', or 'Reject'")

print(" Pydantic models created successfully")

 Pydantic models created successfully


In [16]:
# Cell 6 - Prompts for each pipeline step

# Prompt 1: Skill Extraction
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert HR assistant. Extract structured information from resumes.
    Rules:
    - ONLY extract skills explicitly mentioned in the resume
    - Do NOT invent or assume any information
    - If something is missing, use empty list [] or 0

    Output must be valid JSON with these exact keys: skills, experience_years, tools"""),
    ("human", "Resume:\n{resume_text}")
])

# Prompt 2: Matching
match_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are comparing a candidate's profile with job requirements.
    Be objective and factual. Only mark skills as matched if they clearly appear.

    Experience fit guidelines:
    - 'low': candidate has < 50% of required experience
    - 'good': candidate meets experience requirement
    - 'exceeds': candidate has > 150% of required experience"""),
    ("human", """CANDIDATE PROFILE:
Skills: {skills}
Experience: {experience_years} years
Tools: {tools}

JOB REQUIREMENTS:
{job_requirements}

Output JSON with keys: matched_skills, missing_skills, experience_fit""")
])

print(" Prompts created successfully")

 Prompts created successfully


In [17]:
# Cell 7 - Create LangChain pipelines

# Chain 1: Extraction
extract_chain = extract_prompt | llm | JsonOutputParser()

# Chain 2: Matching
match_chain = match_prompt | llm | JsonOutputParser()

print(" LangChain pipelines created")

 LangChain pipelines created


In [18]:
# Cell 8 - Job Description and Resumes

JOB_DESCRIPTION = """
DATA SCIENTIST ROLE REQUIREMENTS:
Required Skills:
- Python
- SQL
- Machine Learning
- Statistics
- Data Visualization

Experience: 2-4 years

Tools: Pandas, Scikit-learn, Jupyter
"""

# STRONG CANDIDATE - Perfect match
RESUME_STRONG = """
EXPERIENCE:
Senior Data Scientist | Tech Corp | 2021-Present (4 years)
- Built ML models using Python and Scikit-learn
- Created dashboards with data visualization tools
Data Analyst | DataSoft | 2019-2021 (2 years)

SKILLS:
Python, SQL, Machine Learning, Statistics, Data Visualization, Deep Learning
Tools: Pandas, Scikit-learn, Jupyter, TensorFlow, AWS
Education: MS in Data Science, Stanford University
"""

# AVERAGE CANDIDATE - Partial match
RESUME_AVERAGE = """
EXPERIENCE:
Data Analyst | MidCorp | 2022-Present (2 years)
- Analyzed data using Python and SQL
- Created Excel reports and dashboards

SKILLS:
Python, SQL, Basic Statistics, Excel
Tools: Pandas, Excel, Tableau
Education: BS in Economics
"""

# WEAK CANDIDATE - Poor match
RESUME_WEAK = """
EXPERIENCE:
Data Intern | SmallStartup | 2024 (3 months)
- Assisted with data entry
- Learned basic Python

SKILLS:
Excel, Basic Python
Tools: None
Education: BA in Business Administration
"""

print("Test data loaded")
print(f"Job: Data Scientist")
print(f"Resumes: Strong, Average, Weak")

Test data loaded
Job: Data Scientist
Resumes: Strong, Average, Weak


In [19]:
# Cell 9 - Pipeline implementation

def extract_skills(resume_text: str) -> dict:
    """Step 1: Extract skills, experience, tools from resume"""
    try:
        result = extract_chain.invoke({"resume_text": resume_text})
        # Validate required fields
        return {
            "skills": result.get("skills", []),
            "experience_years": result.get("experience_years", 0),
            "tools": result.get("tools", [])
        }
    except Exception as e:
        print(f" Extraction error: {e}")
        return {"skills": [], "experience_years": 0, "tools": []}

def match_with_job(extracted: dict, job_req: str) -> dict:
    """Step 2: Match candidate profile with job requirements"""
    try:
        result = match_chain.invoke({
            "skills": extracted.get("skills", []),
            "experience_years": extracted.get("experience_years", 0),
            "tools": extracted.get("tools", []),
            "job_requirements": job_req
        })
        return result
    except Exception as e:
        print(f" Matching error: {e}")
        return {"matched_skills": [], "missing_skills": [], "experience_fit": "low"}

def calculate_score(match_result: dict) -> dict:
    """Step 3: Calculate fit score based on match data"""
    # Define required skills from job description
    required_skills = ["Python", "SQL", "Machine Learning", "Statistics"]

    matched_count = len(match_result.get("matched_skills", []))
    missing_count = len(match_result.get("missing_skills", []))
    exp_fit = match_result.get("experience_fit", "low")

    # Calculate using formula
    skill_points = min(matched_count * 15, 75)

    exp_points = {"good": 25, "low": 10, "exceeds": 20}.get(exp_fit, 10)

    penalty = missing_count * 10

    raw_score = skill_points + exp_points - penalty
    final_score = max(0, min(100, raw_score))

    # Generate reasoning
    reasoning = f"Matched {matched_count} required skills ({skill_points}/75 pts). "
    reasoning += f"Experience fit: {exp_fit} ({exp_points}/25 pts). "
    if penalty > 0:
        reasoning += f"Penalty of -{penalty} for {missing_count} missing skills."

    # Recommendation based on score
    if final_score >= 75:
        recommendation = "Strong Hire"
    elif final_score >= 50:
        recommendation = "Consider"
    else:
        recommendation = "Reject"

    return {
        "score": final_score,
        "reasoning": reasoning,
        "recommendation": recommendation
    }

def screen_resume(resume_text: str, candidate_name: str = "") -> dict:
    """Complete pipeline: Resume → Score + Explanation + Tracing"""

    print(f"\n{'='*60}")
    print(f" SCREENING: {candidate_name}")
    print(f"{'='*60}")

    # Step 1: Extract
    print(" Step 1: Extracting skills & experience...")
    extracted = extract_skills(resume_text)
    print(f"    Skills: {extracted['skills'][:5]}{'...' if len(extracted['skills']) > 5 else ''}")
    print(f"    Experience: {extracted['experience_years']} years")
    print(f"    Tools: {extracted['tools']}")

    # Step 2: Match
    print("\n🔗 Step 2: Matching with job requirements...")
    match_result = match_with_job(extracted, JOB_DESCRIPTION)
    print(f"  Matched: {match_result['matched_skills']}")
    print(f"  Missing: {match_result['missing_skills']}")
    print(f"  Experience fit: {match_result['experience_fit']}")

    # Step 3: Score
    print("\n Step 3: Calculating final score...")
    score_result = calculate_score(match_result)
    print(f"    Score: {score_result['score']}/100")
    print(f"    Reasoning: {score_result['reasoning']}")
    print(f"    Recommendation: {score_result['recommendation']}")

    return {
        "candidate": candidate_name,
        "extracted": extracted,
        "match": match_result,
        "score": score_result
    }

print(" Pipeline functions defined")

 Pipeline functions defined


In [20]:
# Cell 10 - Run for all three candidates
print(" STARTING AI RESUME SCREENING SYSTEM WITH GROQ")
print("="*60)

results = {}

# Screen each candidate
results["Strong"] = screen_resume(RESUME_STRONG, "Strong Candidate (4+ yrs experience)")
results["Average"] = screen_resume(RESUME_AVERAGE, "Average Candidate (2 yrs experience)")
results["Weak"] = screen_resume(RESUME_WEAK, "Weak Candidate (3 months internship)")

# Summary
print("\n" + "="*60)
print(" FINAL SUMMARY TABLE")
print("="*60)
print(f"{'Candidate':<15} {'Score':<8} {'Recommendation':<15}")
print("-"*45)
for name, data in results.items():
    print(f"{name:<15} {data['score']['score']}/100   {data['score']['recommendation']:<15}")

 STARTING AI RESUME SCREENING SYSTEM WITH GROQ

 SCREENING: Strong Candidate (4+ yrs experience)
 Step 1: Extracting skills & experience...
    Skills: ['Python', 'SQL', 'Machine Learning', 'Statistics', 'Data Visualization']...
    Experience: 6 years
    Tools: ['Pandas', 'Scikit-learn', 'Jupyter', 'TensorFlow', 'AWS']

🔗 Step 2: Matching with job requirements...
  Matched: ['Python', 'SQL', 'Machine Learning', 'Statistics', 'Data Visualization']
  Missing: []
  Experience fit: exceeds

 Step 3: Calculating final score...
    Score: 95/100
    Reasoning: Matched 5 required skills (75/75 pts). Experience fit: exceeds (20/25 pts). 
    Recommendation: Strong Hire

 SCREENING: Average Candidate (2 yrs experience)
 Step 1: Extracting skills & experience...
    Skills: ['Python', 'SQL', 'Basic Statistics', 'Excel']
    Experience: 2 years
    Tools: ['Pandas', 'Excel', 'Tableau']

🔗 Step 2: Matching with job requirements...
  Matched: ['Python', 'SQL', 'Statistics']
  Missing: ['Machine L